# Top-8 kid category-level cosine similarity

This notebook computes category-level cosine similarity between kids using per-kid mean category embeddings.

Example: average `cat` embedding in subject A vs average `cat` embedding in subject B.

Outputs are saved under `analysis/manuscript-2026/main_results_valid129s_04302026/results/`.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import combinations
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/home/j7yang/babyview-projects/vss2026/object-detection')
ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
PREPRINT_DIR = ANALYSIS_DIR / 'manuscript-2026'
OUTPUT_TAG = 'valid129s_04302026'
MAIN_RESULTS_DIR = PREPRINT_DIR / f'main_results_{OUTPUT_TAG}'
RESULTS_DIR = MAIN_RESULTS_DIR / 'results'
FIGURES_DIR = MAIN_RESULTS_DIR / 'figures'

TRAJECTORY_CSV = ANALYSIS_DIR / 'individual_analyses' / 'developmental_trajectory_rdms_clip' / 'trajectory_correlations.csv'
ORDER_CSV = RESULTS_DIR / 'bv_things_rdm_order_bv_semantic_clip_filtered-0.27_valid129.csv'
EMBEDDINGS_DIR = Path('/data2/dataset/babyview/868_hours/outputs/yoloe_cdi_embeddings/clip_embeddings_new')

# Match notebook 04 conventions and speed knobs.
EMBED_LOAD_MAX_WORKERS = max(4, min(16, (os.cpu_count() or 8)))
EMBED_PARALLEL_CATEGORIES = True
EMBED_CATEGORY_MAX_WORKERS = min(8, max(2, (os.cpu_count() or 4) // 2))
MAX_EXEMPLARS_PER_CATEGORY: int | None = 128
EXEMPLAR_SUBSAMPLE_SEED = 42
ZSCORE_EMBEDDINGS_ACROSS_CATEGORIES = True

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Trajectory CSV: {TRAJECTORY_CSV} (exists: {TRAJECTORY_CSV.exists()})')
print(f'Order CSV: {ORDER_CSV} (exists: {ORDER_CSV.exists()})')
print(f'Embeddings dir: {EMBEDDINGS_DIR} (exists: {EMBEDDINGS_DIR.exists()})')
print(f'ZSCORE_EMBEDDINGS_ACROSS_CATEGORIES: {ZSCORE_EMBEDDINGS_ACROSS_CATEGORIES}')
print(f'MAX_EXEMPLARS_PER_CATEGORY: {MAX_EXEMPLARS_PER_CATEGORY}')
print(f'EMBED_PARALLEL_CATEGORIES / workers: {EMBED_PARALLEL_CATEGORIES} / {EMBED_CATEGORY_MAX_WORKERS}')


In [ ]:
def load_ordered_categories(order_csv: Path) -> list[str]:
    order_df = pd.read_csv(order_csv).sort_values('position').reset_index(drop=True)
    return order_df['category'].astype(str).str.strip().str.lower().tolist()


def get_top8_subjects(trajectory_csv: Path) -> list[str]:
    # Match 04_individual_rdms.ipynb convention exactly.
    traj_df = pd.read_csv(trajectory_csv)
    traj_df['subject_id'] = traj_df['subject_id'].astype(str).str.zfill(8)

    if 'density' not in traj_df.columns:
        if {'n_categories_younger', 'n_categories_older'}.issubset(set(traj_df.columns)):
            traj_df['density'] = traj_df['n_categories_younger'] + traj_df['n_categories_older']
        else:
            raise ValueError('trajectory_correlations.csv must have either density or younger/older count columns.')

    ranked_density_df = (
        traj_df[['subject_id', 'density']]
        .drop_duplicates(subset=['subject_id'])
        .sort_values('density', ascending=False)
        .reset_index(drop=True)
    )
    return ranked_density_df.head(8)['subject_id'].tolist()


def parse_subject_id_from_embedding_stem(stem: str) -> str | None:
    # Match notebook 04 parser: support both YOLOE and legacy subject-first stems.
    parts = stem.split('_')
    if len(parts) >= 6:
        sid = parts[2].strip()
        if sid.isdigit():
            return sid.zfill(8)
    if parts and parts[0].strip().isdigit():
        return parts[0].strip().zfill(8)
    return None


def _stem_quick_maybe_subject(stem: str, allow: set[str]) -> bool:
    for sid in allow:
        if stem.startswith(f'{sid}_') or f'_{sid}_' in stem:
            return True
    return False


def _iter_category_npy_paths(cat_dir: Path, subject_ids_filter: set[str] | None):
    if subject_ids_filter is None:
        yield from cat_dir.glob('*.npy')
        return

    with os.scandir(cat_dir) as it:
        for entry in it:
            if not entry.is_file(follow_symlinks=False):
                continue
            name = entry.name
            if not name.endswith('.npy'):
                continue
            stem = name[:-4]
            if not _stem_quick_maybe_subject(stem, subject_ids_filter):
                continue
            yield Path(entry.path)


def _load_npy_task(item: tuple[Path, str]):
    path, subject_id = item
    try:
        return subject_id, np.load(path)
    except Exception:
        return None


def _l2_normalize_vec(v: np.ndarray) -> np.ndarray:
    x = np.asarray(v, dtype=np.float64).ravel()
    n = np.linalg.norm(x)
    return x / n if n > 0 else x


def zscore_rows_across_categories(X: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    mu = X.mean(axis=0)
    sig = X.std(axis=0)
    return (X - mu) / (sig + eps)


def zscore_subject_embedding_dict(cat_to_vec: dict[str, np.ndarray], eps: float = 1e-10) -> dict[str, np.ndarray]:
    if len(cat_to_vec) < 2:
        return dict(cat_to_vec)
    cats = sorted(cat_to_vec.keys())
    X = np.stack([np.asarray(cat_to_vec[c], dtype=np.float64).ravel() for c in cats], axis=0)
    Xz = zscore_rows_across_categories(X, eps=eps)
    return {c: Xz[i] for i, c in enumerate(cats)}


def _load_vectors_for_one_category(
    cat_dir: Path,
    top8_set: set[str],
    inner_workers: int,
) -> tuple[str, dict[str, list[np.ndarray]]]:
    category = cat_dir.name.strip().lower()
    sid_to_vecs: dict[str, list[np.ndarray]] = defaultdict(list)
    batch: list[tuple[Path, str]] = []

    for emb_file in _iter_category_npy_paths(cat_dir, top8_set):
        subject_id = parse_subject_id_from_embedding_stem(emb_file.stem)
        if subject_id is None or subject_id not in top8_set:
            continue
        batch.append((emb_file, subject_id))

    if not batch:
        return category, {}

    iw = max(1, inner_workers)
    if iw <= 1 or len(batch) < 12:
        for path, subject_id in batch:
            r = _load_npy_task((path, subject_id))
            if r is None:
                continue
            sid, emb = r
            sid_to_vecs[sid].append(emb)
    else:
        chunksize = max(1, min(256, len(batch) // (iw * 4)))
        with ThreadPoolExecutor(max_workers=iw) as pool:
            for r in pool.map(_load_npy_task, batch, chunksize=chunksize):
                if r is None:
                    continue
                sid, emb = r
                sid_to_vecs[sid].append(emb)

    return category, dict(sid_to_vecs)


def load_top8_category_means(
    embeddings_dir: Path,
    ordered_categories: list[str],
    top8_subjects: list[str],
) -> dict[str, dict[str, np.ndarray]]:
    allowed = set(ordered_categories)
    top8_set = set(top8_subjects)
    subject_category_lists: dict[str, dict[str, list[np.ndarray]]] = defaultdict(lambda: defaultdict(list))

    category_dirs = sorted(
        [p for p in embeddings_dir.iterdir() if p.is_dir() and p.name.strip().lower() in allowed],
        key=lambda p: p.name.lower(),
    )

    if EMBED_PARALLEL_CATEGORIES and len(category_dirs) >= 4 and EMBED_CATEGORY_MAX_WORKERS > 1:
        # Parallelize category folders, keep per-folder loads sequential to avoid nested thread storms.
        def _one(cd: Path):
            return _load_vectors_for_one_category(cd, top8_set, 1)

        with ThreadPoolExecutor(max_workers=EMBED_CATEGORY_MAX_WORKERS) as pool:
            futs = {pool.submit(_one, cd): cd for cd in category_dirs}
            for fut in tqdm(as_completed(futs), total=len(category_dirs), desc=f'Loading categories from {embeddings_dir.name}'):
                category, sid_map = fut.result()
                for sid, vecs in sid_map.items():
                    subject_category_lists[sid][category].extend(vecs)
    else:
        for cat_dir in tqdm(category_dirs, desc=f'Loading categories from {embeddings_dir.name}'):
            category, sid_map = _load_vectors_for_one_category(cat_dir, top8_set, EMBED_LOAD_MAX_WORKERS)
            for sid, vecs in sid_map.items():
                subject_category_lists[sid][category].extend(vecs)

    rng = np.random.default_rng(EXEMPLAR_SUBSAMPLE_SEED)
    raw_means: dict[str, dict[str, np.ndarray]] = {}
    for sid, cat_map in tqdm(subject_category_lists.items(), desc='Averaging by subject'):
        raw_means[sid] = {}
        for cat, vecs in cat_map.items():
            if not vecs:
                continue
            kept = vecs
            if MAX_EXEMPLARS_PER_CATEGORY is not None and len(vecs) > MAX_EXEMPLARS_PER_CATEGORY:
                pick = rng.choice(len(vecs), size=MAX_EXEMPLARS_PER_CATEGORY, replace=False)
                kept = [vecs[i] for i in sorted(pick.tolist())]

            arr = np.stack([_l2_normalize_vec(e) for e in kept], axis=0)
            mean_emb = arr.mean(axis=0)
            raw_means[sid][cat] = mean_emb

    subject_category_means: dict[str, dict[str, np.ndarray]] = {}
    for sid, cat_dict in raw_means.items():
        if ZSCORE_EMBEDDINGS_ACROSS_CATEGORIES:
            subject_category_means[sid] = zscore_subject_embedding_dict(cat_dict)
        else:
            subject_category_means[sid] = {}
            for cat, mean_emb in cat_dict.items():
                nrm = np.linalg.norm(mean_emb)
                subject_category_means[sid][cat] = (mean_emb / nrm) if nrm > 0 else mean_emb

    ordered_map: dict[str, dict[str, np.ndarray]] = {}
    for sid in top8_subjects:
        ordered_map[sid] = subject_category_means.get(sid, {})
    return ordered_map


ordered_categories = load_ordered_categories(ORDER_CSV)
top8_subjects = get_top8_subjects(TRAJECTORY_CSV)
subject_category_means = load_top8_category_means(EMBEDDINGS_DIR, ordered_categories, top8_subjects)

print(f'Top-8 subjects: {top8_subjects}')
for sid in top8_subjects:
    print(f'{sid}: {len(subject_category_means[sid])} categories with mean embeddings')


In [ ]:
rows = []
for sid_a, sid_b in combinations(top8_subjects, 2):
    cats_a = set(subject_category_means[sid_a].keys())
    cats_b = set(subject_category_means[sid_b].keys())
    common_cats = sorted(cats_a.intersection(cats_b))
    for cat in common_cats:
        va = subject_category_means[sid_a][cat]
        vb = subject_category_means[sid_b][cat]
        cos = float(cosine_similarity(va.reshape(1, -1), vb.reshape(1, -1))[0, 0])
        rows.append(
            {
                'subject_a': sid_a,
                'subject_b': sid_b,
                'category': cat,
                'cosine_similarity': cos,
            }
        )

pairwise_category_df = pd.DataFrame(rows)
if pairwise_category_df.empty:
    raise RuntimeError('No pairwise category cosines computed. Check paths and available embeddings.')

pairwise_summary_df = (
    pairwise_category_df
    .groupby(['subject_a', 'subject_b'], as_index=False)
    .agg(
        mean_category_cosine=('cosine_similarity', 'mean'),
        median_category_cosine=('cosine_similarity', 'median'),
        std_category_cosine=('cosine_similarity', 'std'),
        n_common_categories=('category', 'count'),
    )
    .sort_values(['mean_category_cosine', 'n_common_categories'], ascending=[False, False])
    .reset_index(drop=True)
)

out_detail_csv = RESULTS_DIR / 'top8_pairwise_category_cosine_detail_clip_valid129.csv'
out_summary_csv = RESULTS_DIR / 'top8_pairwise_category_cosine_summary_clip_valid129.csv'
pairwise_category_df.to_csv(out_detail_csv, index=False)
pairwise_summary_df.to_csv(out_summary_csv, index=False)

print(f'Saved detail CSV: {out_detail_csv}')
print(f'Saved summary CSV: {out_summary_csv}')
display(pairwise_summary_df.head(15))


In [ ]:
subjects = top8_subjects
mat = pd.DataFrame(np.nan, index=subjects, columns=subjects, dtype=float)
for sid in subjects:
    mat.loc[sid, sid] = 1.0

for _, row in pairwise_summary_df.iterrows():
    a = row['subject_a']
    b = row['subject_b']
    v = float(row['mean_category_cosine'])
    mat.loc[a, b] = v
    mat.loc[b, a] = v

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(mat.values, vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(np.arange(len(subjects)))
ax.set_yticks(np.arange(len(subjects)))
ax.set_xticklabels(subjects, rotation=45, ha='right')
ax.set_yticklabels(subjects)
ax.set_title('Top-8 kids: mean category-level cosine similarity (CLIP)')

for i in range(len(subjects)):
    for j in range(len(subjects)):
        val = mat.values[i, j]
        text = '' if np.isnan(val) else f'{val:.2f}'
        ax.text(j, i, text, ha='center', va='center', color='white', fontsize=8)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Mean cosine similarity across common categories')
fig.tight_layout()

out_png = FIGURES_DIR / 'top8_pairwise_category_cosine_heatmap_clip_valid129.png'
out_pdf = FIGURES_DIR / 'top8_pairwise_category_cosine_heatmap_clip_valid129.pdf'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
fig.savefig(out_pdf, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved heatmap: {out_png}')
print(f'Saved heatmap: {out_pdf}')
